# Notebook 04 — Transformer Block

**Goal:** Build a full GPT-style transformer block from the pieces we've already covered.

A decoder block is the repeating unit inside the model:

```
input x
   │
   ▼  LayerNorm
multi-head self-attention
   │
   ▼  residual add
x + attention(x)
   │
   ▼  LayerNorm
feed-forward network
   │
   ▼  residual add
x + FFN(x)
```

---

### Contents
1. [The feed-forward network](#1)
2. [Residuals and layer norm](#2)
3. [A minimal transformer block](#3)
4. [Stacking blocks](#4)
5. [Key takeaways](#5)


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

print('=' * 60)
print('  Notebook 04 — Transformer Block')
print('=' * 60)
print(f'  PyTorch : {torch.__version__}')
print('=' * 60)


<a id='1'></a>
## 1 — The feed-forward network

Attention mixes information **between tokens**.

The feed-forward network (FFN) processes each token **independently** after attention:

$$
\mathrm{FFN}(x) = W_2\,\mathrm{GELU}(W_1 x + b_1) + b_2
$$

Usually the hidden dimension expands to about `4 × d_model` and then contracts back.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = F.gelu(x)
        x = self.linear2(x)
        return x

x = torch.randn(1, 4, 8)
ff = FeedForward(d_model=8, d_ff=32)
out = ff(x)

print('Input shape :', tuple(x.shape))
print('Output shape:', tuple(out.shape))


<a id='2'></a>
## 2 — Residuals and layer norm

Two design choices make deep transformers train well:

- **Residual connection**: the block learns a change to add on top of the input
- **LayerNorm**: keeps activations on a stable scale

GPT-2 uses **pre-norm**, so we normalise *before* attention and *before* the FFN.


In [ ]:
x_small = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
ln = nn.LayerNorm(4)

print('Original vector :', x_small.numpy())
print('After LayerNorm :', ln(x_small).detach().numpy().round(3))


<a id='3'></a>
## 3 — A minimal transformer block


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        batch, seq, _ = x.shape
        Q = self.W_q(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        weights = torch.nan_to_num(weights, nan=0.0)
        out = weights @ V
        out = out.transpose(1, 2).contiguous().view(batch, seq, self.d_model)
        return self.W_o(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask=mask)
        x = x + self.ff(self.ln2(x))
        return x


In [ ]:
d_model = 16
n_heads = 4
d_ff = 64
seq_len = 6

block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(2, seq_len, d_model)
mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
mask = mask.unsqueeze(0).unsqueeze(0)

y = block(x, mask=mask)

print('Input shape :', tuple(x.shape))
print('Output shape:', tuple(y.shape))
print('Shape preserved:', x.shape == y.shape)


<a id='4'></a>
## 4 — Stacking blocks

Because the input and output shapes match, blocks can be stacked one after another.


In [ ]:
stack = nn.ModuleList([
    TransformerBlock(d_model=16, n_heads=4, d_ff=64)
    for _ in range(3)
])

x = torch.randn(1, 6, 16)
for i, block in enumerate(stack):
    x = block(x, mask=mask)
    print(f'After block {i}:', tuple(x.shape))


<a id='5'></a>
## 5 — Key takeaways

- A transformer block combines **attention**, **FFN**, **LayerNorm**, and **residual connections**
- Attention handles token-to-token interaction
- The FFN handles per-token transformation
- Residuals and pre-norm LayerNorm are what make deep stacking practical
- Full GPT-style models are just many of these blocks in sequence

Next, we'll train a tiny language model built from these blocks.
